# Research 16 — Loyalty × VIP × engagement interaction

**Hypothesis:** the positive two-way VIP interactions may be complementary. One combined category could represent high-value player subgroups better than separate pairs. Add a single missing-aware three-way categorical feature, leave the selected LightGBM settings unchanged, and run one deterministic 3-fold OOF test. Score a fixed 80/20 blend and fixed four-way mixture. This is one hypothesis, not a combinatorial sweep. Test is not loaded.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and np.all(probs >= 0)
    row_sums = probs.sum(axis=1, keepdims=True)
    assert np.all(row_sums > 0)
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
p_xgb = load_probs("segment-xgboost-oof.csv")
p_balanced = load_probs("segment-balanced-catboost-oof.csv")
p_previous = load_probs("segment-vip-engagement-four-way-oof.csv")


def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    loyalty = x["loyalty_tier"].fillna("__MISSING__").astype(str)
    engagement = x["engagement_level"].fillna("__MISSING__").astype(str)
    vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
    x["loyalty_vip_engagement"] = loyalty + "__" + vip + "__" + engagement
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Features={X.shape[1]}; categoricals={len(cat_cols)}; interaction=loyalty_vip_engagement")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_interaction = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    for col in cat_cols:
        dtype = pd.CategoricalDtype(categories=sorted(x_fit[col].dropna().unique().tolist()))
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    p_interaction[valid_idx] = model.predict_proba(x_valid)
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_interaction[valid_idx].argmax(1), average='macro'):.6f}",
        flush=True,
    )

p_blend = 0.8 * p_base + 0.2 * p_interaction
p_four_way = 0.7 * p_base + 0.1 * p_xgb + 0.1 * p_balanced + 0.1 * p_interaction
candidates = {
    "Current reference": p_base,
    "Previous best four-way VIP × engagement": p_previous,
    "LightGBM three-way interaction": p_interaction,
    "80/20 current + three-way interaction": p_blend,
    "70/10/10/10 with three-way interaction": p_four_way,
}
folds = list(cv.split(X, y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current reference", [f"fold_{fold}_macro_f1" for fold in range(1, 4)]
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-three-way-interaction-results.csv", index=False)
for suffix, probs in [
    ("three-way-interaction", p_interaction),
    ("three-way-interaction-blend", p_blend),
    ("three-way-interaction-four-way", p_four_way),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-lightgbm-{suffix}-oof.csv", index=False
    )
print(f"Elapsed seconds={perf_counter() - started:.1f}. Training OOF only; Test not loaded.")

Features=52; categoricals=11; interaction=loyalty_vip_engagement


Fold 1: macro F1=0.739250


Fold 2: macro F1=0.741661


Fold 3: macro F1=0.744385


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,Previous best four-way VIP × engagement,0.751386,0.765039,0.749681,0.750712,0.753734,0.000235,0.000891,0.000345
2,LightGBM three-way interaction,0.741775,0.756776,0.739250,0.741661,0.744385,-0.010196,-0.008160,-0.009005
3,80/20 current + three-way interaction,0.750506,0.764428,0.748613,0.750043,0.752824,-0.000832,0.000222,-0.000565
4,70/10/10/10 with three-way interaction,0.751242,0.764922,0.750211,0.750040,0.753450,0.000766,0.000219,0.000060


Elapsed seconds=58.5. Training OOF only; Test not loaded.


## Decision

Check whether the additional joint category improves on the pairwise interactions. Do not promote small differences from these repeatedly used folds without independent validation.